# Seed Ranking

Lightweight ranking of training seeds by size-dependent foraging performance.
Run this first to identify the best seeds; use `seed_analysis.ipynb` for full characterisation.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
import utils_seeds as us


# ConsNoise20260608dynamicT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU 20
# ConsNoise20260609dynamicT5MFO0.0FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU 20
# ConsNoise20260609dynamicT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU 20
# ConsNoise20260610fracT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU 20

# ConsNoise20260610dynamicT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU 16

# ── CONFIGURE HERE ────────────────────────────────────────────────────────────
RESULTS_DIR       = Path("/home/satsingh/cluster_lab/satsingh/marl_fish_storage/results")
# GROUP_FOLDER_NAME = "ConsNoise20260608dynamicT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU" # 20 seeds; best seed 8, then seed 15
# GROUP_FOLDER_NAME = "ConsNoise20260609dynamicT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU" # 20 seeds; best seed 7, then seed 15
# GROUP_FOLDER_NAME = "ConsNoise20260609dynamicT5MFO0.0FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU" # 20 seeds; best seed 16, then seed 15
GROUP_FOLDER_NAME = "ConsNoise20260610fracT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU" # 20 seeds; best seed 14, then seed 15
MAIN_EVAL         = "m1a1k1_patchy_square"
LAST_N            = 5
# ─────────────────────────────────────────────────────────────────────────────

run_dirs = us.discover_run_dirs(RESULTS_DIR, GROUP_FOLDER_NAME)
print(f"Group : {GROUP_FOLDER_NAME}")
print(f"Runs  : {len(run_dirs)}")
for r in run_dirs:
    print(f"  {r.parent.name} / {r.name}")

In [ ]:
df = us.load_ranking_df(run_dirs, main_eval=MAIN_EVAL, last_n=LAST_N)
df = us.rank_seeds(df)
print(f"Loaded {len(df)} seeds.")

In [ ]:
show = ["seed", "food_mean", "food_std", "size_food_slope", "size_food_r2", "size_food_p",
        "theil_mean", "biting_mean", "train_final_reward", "train_final_r_food", "combined_rank"]
show = [c for c in show if c in df.columns]
ranked = df[show].sort_values("combined_rank").reset_index(drop=True)
float_cols = ranked.select_dtypes("float").columns.tolist()
grad_cols  = [c for c in ["size_food_slope", "food_mean", "size_food_r2"] if c in ranked.columns]
display(
    ranked.style
    .background_gradient(cmap="RdYlGn", subset=grad_cols, axis=0)
    .format({c: "{:.3f}" for c in float_cols})
)

In [ ]:
us.plot_ranking_landscape(df)

In [ ]:
# us.plot_ranking_bar(df)

In [ ]:
us.plot_slope_vs_r2(df)

In [ ]:
us.plot_reward_curve(run_dirs)

In [ ]:
us.plot_r_food_curve(run_dirs)

In [ ]:
us.plot_r_bitten_curve(run_dirs)

In [ ]:
import numpy as np
print("=== Recommendation ===")
for _, row in df.sort_values("combined_rank").iterrows():
    marker = "  <- BEST" if row["combined_rank"] == df["combined_rank"].min() else ""
    p_str  = f"{row['size_food_p']:.2g}" if not np.isnan(row.get("size_food_p", float("nan"))) else "N/A"
    print(
        f"Seed {int(row['seed']):2d}: "
        f"slope={row['size_food_slope']:+.3f}  "
        f"R²={row['size_food_r2']:.3f}  "
        f"p={p_str}  "
        f"food={row['food_mean']:.1f}±{row['food_std']:.1f}  "
        f"rank={int(row['combined_rank'])}"
        f"{marker}"
    )